# Chapter 3 — Looking Inside Large Language Models
### Practice Notebook

*Source: Hands-On Large Language Models, Jay Alammar & Maarten Grootendorst (O'Reilly)*

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/HandsOnLLM/Hands-On-Large-Language-Models/blob/main/chapter03/Chapter%203%20-%20Looking%20Inside%20LLMs.ipynb)

---

This chapter looks inside the Transformer architecture: how tokens flow through the model, how the LM head converts hidden states to token probabilities, what the KV cache is, and how self-attention works mathematically. This notebook gives you hands-on exercises for every concept.

---

## Table of Contents

- [Part 1: Loading the LLM and Basic Generation](#part-1-loading-the-llm-and-basic-generation)
- [Part 2: Inspecting the Model Architecture](#part-2-inspecting-the-model-architecture)
  - [Exercise 2.1 — Print and Read the Model Architecture](#exercise-21--print-and-read-the-model-architecture)
  - [Exercise 2.2 — Count the Layers and Identify Dimensions](#exercise-22--count-the-layers-and-identify-dimensions)
- [Part 3: The Manual Forward Pass](#part-3-the-manual-forward-pass)
  - [Exercise 3.1 — Tokenise and Run Through the Transformer Stack](#exercise-31--tokenise-and-run-through-the-transformer-stack)
  - [Exercise 3.2 — Run Through the LM Head](#exercise-32--run-through-the-lm-head)
  - [Exercise 3.3 — Pick the Next Token with Greedy Decoding](#exercise-33--pick-the-next-token-with-greedy-decoding)
- [Part 4: Exploring the Token Probability Distribution](#part-4-exploring-the-token-probability-distribution)
  - [Exercise 4.1 — Apply Softmax and Get Token Probabilities](#exercise-41--apply-softmax-and-get-token-probabilities)
  - [Exercise 4.2 — Top-K Most Probable Next Tokens](#exercise-42--top-k-most-probable-next-tokens)
  - [Exercise 4.3 — How Prompt Wording Changes the Distribution](#exercise-43--how-prompt-wording-changes-the-distribution)
- [Part 5: KV Cache — Timing the Speedup](#part-5-kv-cache--timing-the-speedup)
  - [Exercise 5.1 — Time Generation With and Without Cache](#exercise-51--time-generation-with-and-without-cache)
- [Part 6: Self-Attention From Scratch](#part-6-self-attention-from-scratch)
  - [Exercise 6.1 — Project Inputs into Q, K, V Spaces](#exercise-61--project-inputs-into-q-k-v-spaces)
  - [Exercise 6.2 — Relevance Scoring: Q × Kᵀ / √dₖ](#exercise-62--relevance-scoring-q--kt--dk)
  - [Exercise 6.3 — Softmax to Get Attention Weights](#exercise-63--softmax-to-get-attention-weights)
  - [Exercise 6.4 — Combine Values Using Attention Weights](#exercise-64--combine-values-using-attention-weights)
  - [Exercise 6.5 — Full Attention in One Function](#exercise-65--full-attention-in-one-function)
- [Part 7: Modern Transformer Architecture](#part-7-modern-transformer-architecture)
  - [Exercise 7.1 — Identify Architectural Improvements in Phi-3](#exercise-71--identify-architectural-improvements-in-phi-3)

### [OPTIONAL] — Install packages on Colab

💡 **GPU required** for Parts 1–5. Parts 6–7 run on CPU.

In [ ]:
# %%capture
# !pip install transformers>=4.41.2 accelerate>=0.31.0

---
# Part 1: Loading the LLM and Basic Generation

A Transformer LLM takes text in and generates text out — one token at a time. Each generation step is one forward pass through the model. The pipeline wraps this loop for us.

```
Prompt  →  [Tokenizer]  →  Token IDs  →  [Transformer Blocks × 32]  →  [LM Head]  →  Probabilities  →  Next token
       ↑________ append output token and repeat _______________________________↑
```

**Task:** Load the Phi-3-mini model and tokenizer, create a text-generation pipeline with `max_new_tokens=50` and `do_sample=False`, then generate a response to the gardening email prompt.

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline

# YOUR CODE HERE
# tokenizer = AutoTokenizer.from_pretrained("microsoft/Phi-3-mini-4k-instruct")
# model = AutoModelForCausalLM.from_pretrained(
#     "microsoft/Phi-3-mini-4k-instruct",
#     device_map="cuda",
#     torch_dtype="auto",
#     trust_remote_code=False,
# )
# generator = pipeline(
#     "text-generation",
#     model=model,
#     tokenizer=tokenizer,
#     return_full_text=False,
#     max_new_tokens=50,
#     do_sample=False,
# )

prompt = "Write an email apologizing to Sarah for the tragic gardening mishap. Explain how it happened."

# output = generator(prompt)
# print(output[0]['generated_text'])

---
# Part 2: Inspecting the Model Architecture

Printing the model variable reveals its complete layer structure. Understanding this structure is fundamental to understanding what happens during a forward pass.

## Exercise 2.1 — Print and Read the Model Architecture

**Task:** Print the model. Then read the output and answer:
1. What is the top-level class name?
2. What is the embedding table size? (vocab_size × d_model)
3. How many transformer decoder layers are there?
4. What is the output size of the LM head?

**Expected answers:**
- Class: `Phi3ForCausalLM`
- Embedding: `Embedding(32064, 3072)` → 32,064 tokens, each 3,072-dimensional
- Layers: 32 blocks
- LM head: `Linear(in_features=3072, out_features=32064)` → one logit per vocab token

In [ ]:
# YOUR CODE HERE
# print(model)

## Exercise 2.2 — Count the Layers and Identify Dimensions

**Task:** Without printing the full model again, programmatically extract:
1. The number of transformer layers (`model.model.layers`)
2. The embedding vocabulary size and dimension from `model.model.embed_tokens`
3. The LM head input and output dimensions from `model.lm_head`

Print each answer clearly.

**Hint:**
```python
len(model.model.layers)                          # number of transformer blocks
model.model.embed_tokens.weight.shape            # (vocab_size, d_model)
model.lm_head.weight.shape                       # (vocab_size, d_model)
```

In [ ]:
# YOUR CODE HERE
# num_layers = len(model.model.layers)
# embed_shape = model.model.embed_tokens.weight.shape
# lm_head_shape = model.lm_head.weight.shape

# print(f"Number of transformer blocks: {num_layers}")
# print(f"Embedding table shape: {embed_shape}  → (vocab_size={embed_shape[0]}, d_model={embed_shape[1]})")
# print(f"LM head weight shape: {lm_head_shape}  → (out={lm_head_shape[0]}, in={lm_head_shape[1]})")
# print(f"\nTotal parameters: {sum(p.numel() for p in model.parameters()):,}")

---
# Part 3: The Manual Forward Pass

The pipeline hides what is happening inside. Here we perform the forward pass manually, step by step:

```
Text prompt
    ↓  tokenizer()
Token IDs  [1, 6, ... ]  shape: [batch=1, seq_len]
    ↓  model.model()  ← 32 transformer blocks
Hidden states  shape: [1, seq_len, 3072]
    ↓  model.lm_head()
Logits  shape: [1, seq_len, 32064]  ← one score per vocab token
    ↓  argmax on last position
Next token ID  →  decode  →  text
```

## Exercise 3.1 — Tokenise and Run Through the Transformer Stack

**Task:**
1. Tokenise `"The capital of France is"` and move to CUDA
2. Run through `model.model(input_ids)` to get hidden states
3. Print the shape of `model_output[0]`

**Expected shape:** `torch.Size([1, 6, 3072])`
- `1` = batch size
- `6` = number of tokens in the prompt
- `3072` = d_model (the dimension of Phi-3's hidden state)

In [ ]:
prompt = "The capital of France is"

# YOUR CODE HERE
# Step 1: Tokenise
# input_ids = tokenizer(prompt, return_tensors="pt").input_ids.to("cuda")
# print(f"Input shape: {input_ids.shape}")
# print(f"Number of tokens: {input_ids.shape[1]}")

# Step 2: Run through transformer blocks (NOT including lm_head)
# model_output = model.model(input_ids)

# Step 3: Print hidden states shape
# print(f"\nHidden states shape: {model_output[0].shape}")
# print("[batch_size=1, seq_len=6, d_model=3072]")

## Exercise 3.2 — Run Through the LM Head

The LM head is a single linear layer that projects the 3,072-dimensional hidden state to a 32,064-dimensional logit vector — one logit per token in the vocabulary. The token with the highest logit is the most likely next token.

**Task:**
1. Run `model_output[0]` through `model.lm_head()`
2. Print the output shape

**Expected shape:** `torch.Size([1, 6, 32064])`
- Still `1` = batch size
- Still `6` = number of input tokens (one logit vector per position)
- Now `32064` = vocabulary size (one score per possible next token)

In [ ]:
# YOUR CODE HERE
# lm_head_output = model.lm_head(model_output[0])
# print(f"LM head output shape: {lm_head_output.shape}")
# print("[batch=1, seq_len=6, vocab_size=32064]")
# print("For next token prediction, we only use the last position (index -1)")

## Exercise 3.3 — Pick the Next Token with Greedy Decoding

For next-token prediction we use the logit vector at the **last position** (`index -1`). Greedy decoding picks the token with the highest logit using `argmax`.

**Task:**
1. Extract the logits for the last token position: `lm_head_output[0, -1]`
2. Find the token ID with the highest logit using `.argmax(-1)`
3. Decode that token ID back to text

**Expected output:** `'Paris'`

In [ ]:
# YOUR CODE HERE
# token_id = lm_head_output[0, -1].argmax(-1)
# print(f"Predicted token ID: {token_id}")
# print(f"Decoded token: {tokenizer.decode(token_id)}")

---
# Part 4: Exploring the Token Probability Distribution

The LM head produces raw **logits** — unnormalised scores. Applying **softmax** converts these to a proper probability distribution that sums to 1. The decoding strategy then picks which token to generate from this distribution.

```
Logits:       [ -3.1,  0.2,  4.7,  1.2,  -8.0, ... ]  (32,064 values)
  ↓ softmax
Probabilities: [ 0.001, 0.04, 0.40, 0.13, 0.0001, ... ]  (sums to 1.0)
  ↓ decoding strategy
Token:         'Dear'  (40% probability → greedy picks this)
```

## Exercise 4.1 — Apply Softmax and Get Token Probabilities

**Task:**
1. Tokenise the email prompt: `"Write an email apologizing to Sarah for the tragic gardening mishap. Explain how it happened."`
2. Run the full forward pass (model.model → lm_head)
3. Apply `torch.softmax` to the last position's logits along dimension -1
4. Print:
   - The probability of the top token
   - The sum of all probabilities (should be exactly 1.0)
   - The number of tokens with probability > 1%

In [ ]:
import torch

email_prompt = "Write an email apologizing to Sarah for the tragic gardening mishap. Explain how it happened."

# YOUR CODE HERE
# Step 1: Tokenise
# input_ids = tokenizer(email_prompt, return_tensors="pt").input_ids.to("cuda")

# Step 2: Forward pass
# model_output = model.model(input_ids)
# logits = model.lm_head(model_output[0])[0, -1]   # logits for last position

# Step 3: Softmax
# probs = torch.softmax(logits, dim=-1)

# Step 4: Print stats
# print(f"Top probability: {probs.max().item():.4f}")
# print(f"Sum of all probabilities: {probs.sum().item():.6f}")
# print(f"Tokens with probability > 1%: {(probs > 0.01).sum().item()}")

## Exercise 4.2 — Top-K Most Probable Next Tokens

**Task:** Find and print the top 10 most probable next tokens for the email prompt, with their probabilities.

Use `torch.topk(probs, k=10)` to get the top probabilities and their indices.

**Expected output (approximately):**
```
Rank  Token        Probability
  1   'Dear'         ~40%
  2   'Title'        ~13%
  3   'To'           ~8%
  4   'Hi'           ~2%
  ...
```

In [ ]:
# YOUR CODE HERE
# top_probs, top_ids = torch.topk(probs, k=10)

# print(f"{'Rank':<6} {'Token':<20} {'Probability':>12}")
# print("-" * 42)
# for rank, (prob, token_id) in enumerate(zip(top_probs, top_ids), 1):
#     token_text = repr(tokenizer.decode(token_id))
#     print(f"{rank:<6} {token_text:<20} {prob.item():>11.2%}")

## Exercise 4.3 — How Prompt Wording Changes the Distribution

Small changes in the prompt can dramatically shift the probability distribution over next tokens.

**Task:** Run the same top-5 analysis on these three prompts and compare the distributions:

```python
prompts = [
    "The capital of France is",       # factual completion
    "Once upon a time",               # story beginning
    "def fibonacci(n):",              # code completion
]
```

**Observe:** Is the distribution concentrated (one token dominates) or spread out? Which prompt type leads to the most certain prediction?

In [ ]:
prompts = [
    "The capital of France is",
    "Once upon a time",
    "def fibonacci(n):",
]

# YOUR CODE HERE
# For each prompt:
#   tokenise → forward pass → softmax → top 5 tokens with probs
#   also print entropy or concentration (top1 prob as a measure)

# for p in prompts:
#     input_ids = tokenizer(p, return_tensors="pt").input_ids.to("cuda")
#     model_out = model.model(input_ids)
#     logits = model.lm_head(model_out[0])[0, -1]
#     probs = torch.softmax(logits, dim=-1)
#     top_p, top_ids = torch.topk(probs, 5)
#     print(f"\nPrompt: '{p}'")
#     print(f"Top token confidence: {top_p[0].item():.2%}")
#     for prob, tid in zip(top_p, top_ids):
#         print(f"  {repr(tokenizer.decode(tid)):<20} {prob.item():.2%}")

---
# Part 5: KV Cache — Timing the Speedup

When generating the second token, the model appends the first output token to the prompt and does another full forward pass. Without caching, it recomputes the key and value matrices for all previous tokens from scratch — every single step. The **KV cache** stores these intermediate results so each new step only computes the new token's keys and values.

```
Without cache:   token 1 → compute all. token 2 → recompute all + new. token 3 → recompute all + new ...
With cache:      token 1 → compute all. token 2 → load cached + compute new only. token 3 → same ...
```

HuggingFace Transformers enables the cache by default (`use_cache=True`).

## Exercise 5.1 — Time Generation With and Without Cache

**Task:**
1. Tokenise the long email prompt below and move to CUDA
2. Time `model.generate(input_ids=..., max_new_tokens=100, use_cache=True)` using `%%timeit -n 1`
3. Time the same call with `use_cache=False`
4. In a separate cell, compute and print the speedup ratio

**Expected results (T4 GPU, approx):**
- With cache: ~4–7 seconds
- Without cache: ~20–22 seconds
- Speedup: ~3–5×

In [ ]:
long_prompt = "Write a very long email apologizing to Sarah for the tragic gardening mishap. Explain how it happened."

# YOUR CODE HERE
# input_ids = tokenizer(long_prompt, return_tensors="pt").input_ids.to("cuda")
# print(f"Prompt token count: {input_ids.shape[1]}")

In [ ]:
%%timeit -n 1
# YOUR CODE HERE
# model.generate(input_ids=input_ids, max_new_tokens=100, use_cache=True)

In [ ]:
%%timeit -n 1
# YOUR CODE HERE
# model.generate(input_ids=input_ids, max_new_tokens=100, use_cache=False)

In [ ]:
# YOUR CODE HERE
# Record your timeit results and compute speedup
# time_with_cache = ...     # seconds from %%timeit above
# time_without_cache = ...  # seconds from %%timeit above
# speedup = time_without_cache / time_with_cache
# print(f"With cache:    {time_with_cache:.1f}s")
# print(f"Without cache: {time_without_cache:.1f}s")
# print(f"Speedup:       {speedup:.1f}x")

---
# Part 6: Self-Attention From Scratch

Self-attention is the core innovation of the Transformer. It allows each token to incorporate information from other tokens in the sequence. This part implements it step by step using tiny matrices so every operation is visible.

**The complete attention formula:**

$$\text{Attention}(Q, K, V) = \text{softmax}\!\left(\frac{QK^T}{\sqrt{d_k}}\right)V$$

**The two steps:**
1. **Relevance scoring** — multiply query by all keys to get a score for each previous token
2. **Combining information** — use those scores (as weights) to take a weighted sum of the value vectors

We will use this toy setup (from the book's Figure 3-20 example):

```
Sentence: "Sarah fed the cat because it"
We are processing the token 'it' and want to know which previous token it refers to.
```

*Note: Parts 6 runs on CPU — no GPU required.*

In [ ]:
import torch
import torch.nn.functional as F
import math

# Toy setup: 4 tokens, d_model=4, d_k=4
# Each row is one token's embedding
torch.manual_seed(42)

# 4 tokens: Sarah, fed, the, cat  (we process 'it' as the query)
# shape: [seq_len=4, d_model=4]
token_embeddings = torch.tensor([
    [1.0, 0.2, 0.5, 0.1],   # Sarah
    [0.3, 0.8, 0.1, 0.6],   # fed
    [0.2, 0.1, 0.9, 0.3],   # the
    [0.9, 0.4, 0.2, 0.7],   # cat
])

# Current token 'it' — the query position
# shape: [1, d_model=4]
it_embedding = torch.tensor([[0.5, 0.6, 0.3, 0.8]])

d_model = 4
d_k = 4  # key/query dimension

# Projection matrices (learned during training — we use random ones here)
W_q = torch.randn(d_model, d_k) * 0.1   # query projection
W_k = torch.randn(d_model, d_k) * 0.1   # key projection
W_v = torch.randn(d_model, d_k) * 0.1   # value projection

print("Setup complete.")
print(f"Token embeddings shape:  {token_embeddings.shape}   (4 context tokens, d_model=4)")
print(f"Query token shape:       {it_embedding.shape}   (1 current token, d_model=4)")
print(f"Projection matrix shape: {W_q.shape}   (d_model=4 → d_k=4)")

## Exercise 6.1 — Project Inputs into Q, K, V Spaces

The projection matrices transform the raw token embeddings into three specialised vector spaces:
- **Query (Q)**: what the current token is looking for
- **Keys (K)**: what each previous token is offering
- **Values (V)**: what information each previous token will contribute if selected

**Task:** Compute Q, K, V by matrix-multiplying the embeddings with the projection matrices.

```
Q = it_embedding     @ W_q   → shape [1, d_k=4]  (query for current token)
K = token_embeddings @ W_k   → shape [4, d_k=4]  (keys for all context tokens)
V = token_embeddings @ W_v   → shape [4, d_k=4]  (values for all context tokens)
```

In [ ]:
# YOUR CODE HERE
# Q = it_embedding     @ W_q
# K = token_embeddings @ W_k
# V = token_embeddings @ W_v

# print(f"Q shape: {Q.shape}  (query for 'it')")
# print(f"K shape: {K.shape}  (keys for Sarah, fed, the, cat)")
# print(f"V shape: {V.shape}  (values for Sarah, fed, the, cat)")
# print(f"\nQ (what 'it' is looking for):\n{Q}")

## Exercise 6.2 — Relevance Scoring: Q × Kᵀ / √dₖ

The relevance score tells the model how much attention to pay to each previous token. It is computed as the dot product of the current query with each key, scaled by $\sqrt{d_k}$.

$$\text{scores} = \frac{Q \cdot K^T}{\sqrt{d_k}}$$

**Why divide by $\sqrt{d_k}$?** When $d_k$ is large, dot products grow large in magnitude, pushing the softmax into saturation (near-zero gradients). Dividing by $\sqrt{d_k}$ keeps the values in a stable range.

**Task:** Compute the raw attention scores (before softmax).

```
scores = Q @ K.T / sqrt(d_k)   → shape [1, 4]  (one score per context token)
```

In [ ]:
# YOUR CODE HERE
# scores = Q @ K.T / math.sqrt(d_k)
# print(f"Raw attention scores shape: {scores.shape}")
# print(f"Raw scores (one per context token): {scores}")
# tokens = ['Sarah', 'fed', 'the', 'cat']
# for token, score in zip(tokens, scores[0]):
#     print(f"  {token:<8}: {score.item():.4f}")

## Exercise 6.3 — Softmax to Get Attention Weights

Softmax converts the raw scores into a probability distribution — all weights are positive and sum to 1. This tells the model how much to attend to each previous token.

**Task:** Apply softmax to the scores along dimension -1 to get attention weights.

**Expected shape:** `[1, 4]` — four weights summing to 1.0

The token with the highest weight is the most relevant context for understanding 'it'.

In [ ]:
# YOUR CODE HERE
# attention_weights = F.softmax(scores, dim=-1)
# print(f"Attention weights shape: {attention_weights.shape}")
# print(f"Sum of weights: {attention_weights.sum().item():.6f}  (should be 1.0)")

# tokens = ['Sarah', 'fed', 'the', 'cat']
# print("\nAttention weights (how much 'it' attends to each token):")
# for token, weight in zip(tokens, attention_weights[0]):
#     bar = '█' * int(weight.item() * 40)
#     print(f"  {token:<8}: {weight.item():.4f}  {bar}")

## Exercise 6.4 — Combine Values Using Attention Weights

The final step: multiply each value vector by its attention weight and sum them all up. The result is a single vector for the current token ('it') that has been **enriched with context information** from the other tokens.

$$\text{output} = \text{attention\_weights} \cdot V$$

**Task:** Compute the weighted sum of value vectors.

```
context_vector = attention_weights @ V   → shape [1, d_k=4]
```

This output vector is the attention layer's contribution for the token 'it'. It now contains a blend of information from all previous tokens, weighted by how relevant each one is.

In [ ]:
# YOUR CODE HERE
# context_vector = attention_weights @ V
# print(f"Context vector shape: {context_vector.shape}")
# print(f"Context vector (enriched representation of 'it'):")
# print(context_vector)
# print("\nThis vector is now the input to the feedforward network for this token.")

## Exercise 6.5 — Full Attention in One Function

**Task:** Combine all four steps into a single `scaled_dot_product_attention(query, keys, values)` function.

Then verify it gives the same output as PyTorch's built-in `F.scaled_dot_product_attention`.

**Function signature:**
```python
def scaled_dot_product_attention(Q, K, V):
    """
    Q: [1, d_k]  — query for current token
    K: [n, d_k]  — keys for all context tokens
    V: [n, d_k]  — values for all context tokens
    Returns: [1, d_k] — context-enriched output vector
    """
```

In [ ]:
def scaled_dot_product_attention(Q, K, V):
    """
    Implements attention(Q, K, V) = softmax(QK^T / sqrt(d_k)) @ V
    Q: [1, d_k], K: [n, d_k], V: [n, d_k]
    Returns: [1, d_k]
    """
    # YOUR CODE HERE
    # Step 1: scores = Q @ K.T / sqrt(d_k)
    # Step 2: weights = softmax(scores, dim=-1)
    # Step 3: output = weights @ V
    pass


# Test
# my_output = scaled_dot_product_attention(Q, K, V)
# print(f"My attention output:    {my_output}")

# Compare with PyTorch's built-in
# pytorch_output = F.scaled_dot_product_attention(Q, K, V)
# print(f"PyTorch's attention:    {pytorch_output}")

# Check they match
# match = torch.allclose(my_output, pytorch_output, atol=1e-5)
# print(f"\nOutputs match: {match}")

---
# Part 7: Modern Transformer Architecture

The original 2017 Transformer paper introduced the core ideas, but modern LLMs like Phi-3 and Llama 3 include several important improvements:

| Component | Original (2017) | Modern (2024) |
|---|---|---|
| Normalisation | LayerNorm (post-attention) | RMSNorm (pre-attention) |
| Attention | Multi-head attention | Grouped-query attention (GQA) |
| Activation | ReLU in FFN | SiLU/SwiGLU in FFN |
| Positional encoding | Absolute (added at start) | RoPE (added inside attention) |

## Exercise 7.1 — Identify Architectural Improvements in Phi-3

**Task:** From the printed model architecture (Exercise 2.1), find and print evidence of each modern improvement listed in the table above. Answer these specific questions:

1. **RMSNorm**: What is the layer class name for normalisation in Phi-3? Print the first normalisation layer.
2. **GQA (Grouped-Query Attention)**: Look at the `qkv_proj` linear layer dimensions. The output is `9216`. With 32 attention heads and d_model=3072, what does 9216 tell you about the number of KV heads?
3. **SiLU activation**: Find the MLP activation function in the printed model.
4. **RoPE**: What is the class name for Phi-3's positional embedding component?

**Hint for question 2:**
- In full multi-head attention: `qkv_proj` output = 3 × d_model = 3 × 3072 = 9216 (Q, K, V each have 32 heads)
- In grouped-query attention: Q has 32 heads but K and V share fewer heads → total `< 3 × d_model`
- Phi-3's `qkv_proj` output is 9216 → what does this tell you?

**Hint for looking up config:**
```python
model.config.num_attention_heads    # number of query heads
model.config.num_key_value_heads    # number of KV heads (GQA)
```

In [ ]:
# YOUR CODE HERE

# 1. RMSNorm
# first_block = model.model.layers[0]
# print("Normalisation layer:", first_block.input_layernorm)

# 2. Grouped-Query Attention
# n_q_heads = model.config.num_attention_heads
# n_kv_heads = model.config.num_key_value_heads
# print(f"Query heads: {n_q_heads}")
# print(f"KV heads:    {n_kv_heads}")
# print(f"GQA ratio:   {n_q_heads // n_kv_heads} query heads per KV head")

# 3. SiLU activation
# print("MLP activation:", first_block.mlp.activation_fn)

# 4. RoPE
# print("Positional embedding:", first_block.self_attn.rotary_emb)

**Your observations:**

1. Normalisation type in Phi-3: *(write what you found)*
2. Number of KV heads vs query heads: *(write what you found and what it implies about memory savings)*
3. Activation function: *(write what you found)*
4. Positional embedding type: *(write what you found)*

---
## Chapter 3 Summary

You have now implemented or directly observed every major concept from the chapter:

| Concept | What you built/observed | Key insight |
|---|---|---|
| Autoregressive generation | Pipeline + manual forward pass | One forward pass per token |
| Model architecture | `print(model)`, counting layers | 32 blocks, 3072 d_model, 32064 vocab |
| Forward pass shapes | `[1,6,3072]` → `[1,6,32064]` | Hidden states → logits → pick token |
| Probability distribution | Softmax on logits, top-k | 'Dear' at ~40% for email prompt |
| KV cache | Timing with/without cache | ~3-5× speedup from caching |
| Self-attention | Q/K/V projections, scores, weights, output | Two steps: score relevance, combine values |
| Modern architecture | GQA, RMSNorm, SiLU, RoPE in Phi-3 | Each change improves speed or quality |